In [1]:
import pandas as pd
import os
#os.chdir('/content/drive/MyDrive/name/IMP-OIC-Windowing')
from utils.extractframes import FrameExtractor
import graphene
from gpt_ask import run_gpt
import re
main = pd.read_csv('/.../.../Developer/IMP-OIC/LifeQA/for_eval/wo_st/OIC_gpt4_wo_st1.csv')

context = str(main['OIC_context'][0])
regex_list = [r"\n[A-Za-z0-9_]+ has hair_[A-Za-z0-9_]+",r"\n[A-Za-z0-9_]+ has mouth_[A-Za-z0-9_]+",r"\n[A-Za-z0-9_]+ has [A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has ear_[A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has head_[A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has nose_[A-Za-z0-9_]+", r"\nmouth_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\nhair_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\nhead_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\neye_[A-Za-z0-9_]+ of [A-Za-z0-9_]+", r"\nnose_[A-Za-z0-9_]+ of [A-Za-z0-9_]+",  r"\n[A-Za-z0-9_]+ has arm_[A-Za-z0-9_]+", r"\n[A-Za-z0-9_]+ has face_[A-Za-z0-9_]+", "[0-9]+ seconds:\nAfter"]

context_list = set(context.split('\n'))
for regex in regex_list:
    for c in context_list:
        replaced_text = re.sub(regex,'', context, re.MULTILINE)
        context = replaced_text

main['OIC_context'] = context


main['subtitle'] = main['subtitle'].fillna('_') 
main = main.loc[main['OIC_context']!= '_']
main['OIC_answer'] = main['OIC_answer'].fillna('_') 
main = main.loc[:, ~main.columns.str.contains('^Unnamed')]
main = main.loc[main['OIC_answer'] == '_']
main.info

<bound method DataFrame.info of      q_id                                           question answer_type  \
0    1011                    How many people are in the car?           V   
1    1012                       What color is the man's hat?           V   
2    1013                                    Where are they?           B   
3    1014                              Where are they going?           L   
4    1015                Where does she want to go to first?           L   
..    ...                                                ...         ...   
619  2282                How many full glasses of water are?           B   
620  2283                            What is the boy's name?           L   
621  2284                           How many cups are there?           V   
622  2285  Does James think that there is the same amount...           L   
623  2286  Does James think that there is the same amount...           L   

                                     Answers     video_

In [2]:
import ast
q_ids = main['q_id'].unique()
i=0
for qid in q_ids:
  #if 'Seq' in q:
    que = main.query("q_id=={}".format(qid))
    question = que['question'].values[0]
    answer_id = que["ans_idx"].values[0]
    subtitle = que['subtitle'].values[0]
    prompt = que['OIC_context'].values[0]
    options = ast.literal_eval(que['Answers'].values[0])
    choice_string = ''

    choice_string = "0: {}, 1: {}, 2: {}, 3: {}".format(options[0], options[1], options[2], options[3])
    
    formatted_question = question+ 'Guess the most likely answer among these options: '+choice_string+' Respond only with a single number between 0 and 3. Do not produce any other output. If enough information is not given, still make a random guess to result in one out of the given options.'
    print('-'*100)
    print(i)
    #print(formatted_question)
    #response = run_gpt(prompt+'\n subtitle: '+subtitle, formatted_question)
    response = run_gpt(prompt, formatted_question)
    main.loc[main['q_id'] == qid, 'OIC_answer'] = ''
    main.loc[main['q_id'] == qid, 'OIC_question'] = formatted_question
    OIC_answer = response
    print('OIC question: {}'.format(formatted_question))
    print(choice_string)
    print('OIC answer: {}'.format(OIC_answer))
    if len(OIC_answer)>1:
      main.loc[main['q_id'] == qid, 'Match'] = OIC_answer
    else:
      if int(OIC_answer) == int(answer_id):
          main.loc[main['q_id'] == qid, 'Match'] = 'Correct'
          print('correct')
      else:
          print('wrong')
          main.loc[main['q_id'] == qid, 'Match'] = 'Wrong'
    
    main.to_csv('LifeQA/for_eval/OIC_gpt4_wo_st1.csv')
    i=i+1
    

----------------------------------------------------------------------------------------------------
0
OIC question: How many people are in the car?Guess the most likely answer among these options: 0: 11, 1: 4, 2: 2, 3: 7 Respond only with a single number between 0 and 3. Do not produce any other output. If enough information is not given, still make a random guess to result in one out of the given options.
0: 11, 1: 4, 2: 2, 3: 7
OIC answer: 2
correct
----------------------------------------------------------------------------------------------------
1
OIC question: What color is the man's hat?Guess the most likely answer among these options: 0: silver, 1: golden, 2: blue, 3: red Respond only with a single number between 0 and 3. Do not produce any other output. If enough information is not given, still make a random guess to result in one out of the given options.
0: silver, 1: golden, 2: blue, 3: red
OIC answer: 1
wrong
---------------------------------------------------------------

import re

list_answers = []
# Read the video description from a text file
with open('/.../.../Developer/IMP-OIC/LifeQA/datatables/output.txt', 'r') as file:
    
    for line in file:
        match = re.search(r'OIC answer: (.+?)(?:\n|$)', line)
        
        if match:
            answer = match.group(1)
            list_answers.append(answer)
            print(answer)

count=505  
for l in list_answers:       
    
    
    main.loc[main.index == count, 'OIC_answer'] = l
    
    count+=1

main.to_csv('LifeQA/for_eval/OIC_gpt4_wo_st_updated.csv')

main.loc[main.index == 1796, 'OIC_answer']

new_pd = pd.read_csv('/.../.../Developer/IMP-OIC/LifeQA/for_eval/OIC_gpt4_wo_st_updated.csv')
new_pd.loc[508,:]['OIC_answer']

print(list_answers)